In [16]:
# 1

import numpy as np
import pandas as pd
import yfinance as yf
import os

rng = np.random.default_rng(0)
pd.set_option("display.width", 200)

TICKERS = {"VIX": "^VIX", "SPX": "^GSPC", "OVX": "^OVX", "GVZ": "^GVZ"}
SECTORS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "XLRE", "XLC"]
SEC9 = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]

WIN, MINP = 60, 40
Z, F = 1.5, 1.0
HORIZONS = [1, 2, 3, 4, 5]
N_PERM = 5000
ROLL_W = 750

PERIODS = [("1990-1999", 1990, 1999), ("2000-2009", 2000, 2009),
           ("2010-2019", 2010, 2019), ("2020-2026", 2020, 2026),
           ("2023-2026", 2023, 2026)]


def welch(a, b):
    a, b = a.dropna(), b.dropna()
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1) / na, b.var(ddof=1) / nb
    t = (a.mean() - b.mean()) / np.sqrt(va + vb)
    return a.mean(), b.mean(), t, na, nb


def perm_diff(y, sig, n_iter=N_PERM):
    y = np.asarray(y, dtype=float)
    sig = np.asarray(sig, dtype=bool)
    ok = ~np.isnan(y)
    obs = np.nanmean(y[sig & ok]) - np.nanmean(y[~sig & ok])
    n = len(y)
    null = np.empty(n_iter)
    for i in range(n_iter):
        sh = np.roll(sig, rng.integers(1, n))
        null[i] = np.nanmean(y[sh & ok]) - np.nanmean(y[~sh & ok])
    mu, sd = null.mean(), null.std()
    return obs, sd, (obs - mu) / sd, (np.abs(null - mu) >= abs(obs - mu)).mean()


def compare(y, sig_mask, base_mask, scale=1.0, perm=True):
    y = pd.Series(y) if not isinstance(y, pd.Series) else y
    ma, mb, t, na, nb = welch(y[sig_mask], y[base_mask])
    out = {"n_sig": na, "n_base": nb, "signal": ma * scale,
           "base": mb * scale, "diff": (ma - mb) * scale, "t_welch": t}
    if perm:
        _, _, z, p = perm_diff(y.values, sig_mask)
        out["perm_z"], out["p_perm"] = z, p
    return out

In [17]:
# 2

os.makedirs("cache", exist_ok=True)
CACHE = "cache/prices.parquet"


def fetch(tickers, start="1990-01-01"):
    raw = yf.download(list(tickers), start=start, auto_adjust=True,
                      progress=False, group_by="column")
    return raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw


if os.path.exists(CACHE):
    px_raw = pd.read_parquet(CACHE)
else:
    a = fetch(list(TICKERS.values())).rename(columns={v: k for k, v in TICKERS.items()})
    b = fetch(SECTORS)
    px_raw = a.join(b, how="outer").sort_index()
    px_raw.index = pd.to_datetime(px_raw.index).tz_localize(None)
    px_raw.to_parquet(CACHE)

px = px_raw[px_raw["SPX"].notna()].copy()

cov = pd.DataFrame({
    "start": px.apply(lambda s: s.first_valid_index()),
    "end": px.apply(lambda s: s.last_valid_index()),
    "n_obs": px.notna().sum(),
    "n_gap": px.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum()
                      if s.first_valid_index() is not None else np.nan),
})
print("[coverage]")
print(cov.to_string())

print("\n[trading days per year]")
yr = px.groupby(px.index.year).size()
print(yr.to_string())

r = np.log(px[["VIX", "SPX"]]).diff().dropna()
print("\n[validation]")
print("rows (SPX calendar):", len(px), "|", px.index.min().date(), "~", px.index.max().date())
print("corr(dlogVIX, dlogSPX):", round(r["VIX"].corr(r["SPX"]), 3))
print("index monotonic:", px.index.is_monotonic_increasing, "| duplicates:", int(px.index.duplicated().sum()))

[coverage]
            start        end  n_obs  n_gap
Ticker                                    
SPX    1990-01-02 2026-07-24   9207      0
GVZ    2008-06-03 2026-07-24   4564      0
OVX    2007-05-10 2026-07-24   4832      0
VIX    1990-01-02 2026-07-24   9207      0
XLB    1998-12-22 2026-07-24   6938      0
XLC    2018-06-19 2026-07-24   2035      0
XLE    1998-12-22 2026-07-24   6938      0
XLF    1998-12-22 2026-07-24   6938      0
XLI    1998-12-22 2026-07-24   6938      0
XLK    1998-12-22 2026-07-24   6938      0
XLP    1998-12-22 2026-07-24   6938      0
XLRE   2015-10-08 2026-07-24   2713      0
XLU    1998-12-22 2026-07-24   6938      0
XLV    1998-12-22 2026-07-24   6938      0
XLY    1998-12-22 2026-07-24   6938      0

[trading days per year]
Date
1990    253
1991    253
1992    254
1993    253
1994    252
1995    252
1996    254
1997    253
1998    252
1999    252
2000    252
2001    248
2002    252
2003    252
2004    252
2005    252
2006    251
2007    251
2008    253


In [18]:
# 3

d = pd.DataFrame(index=px.index)
d["spx_ret"] = np.log(px["SPX"]).diff()
d["spx_sd"] = d["spx_ret"].rolling(WIN, min_periods=MINP).std().shift(1)
d["spx_z"] = d["spx_ret"] / d["spx_sd"]

for v in ["VIX", "OVX", "GVZ"]:
    ch = np.log(px[v]).diff()
    d[f"{v.lower()}_ch"] = ch
    d[f"{v.lower()}_z"] = ch / ch.rolling(WIN, min_periods=MINP).std().shift(1)

base = d.dropna(subset=["vix_z", "spx_z"]).copy()
hi = base["vix_z"] > Z
fl = base["spx_z"].abs() < F

grp = pd.Series(index=base.index, dtype=object)
grp[hi & fl] = "hi_flat"
grp[hi & ~fl] = "hi_move"
grp[~hi & fl] = "lo_flat"
grp[~hi & ~fl] = "lo_move"

SIG = (grp == "hi_flat").values
BASE = (grp == "lo_flat").values

n = len(base)
print("[sample]", base.index.min().date(), "~", base.index.max().date(), "| n =", n)
print("\n[2x2 groups]")
print(grp.value_counts().to_string())

print("\n[signal counts by threshold]")
zs, fs = [1.5, 1.75, 2.0, 2.5], [0.25, 0.5, 0.75, 1.0]
cnt = pd.DataFrame(index=[f"z>{z}" for z in zs], columns=[f"|s|<{f}" for f in fs], dtype=float)
rat = cnt.copy()
for z in zs:
    ps = (base["vix_z"] > z).mean()
    for f in fs:
        pf = (base["spx_z"].abs() < f).mean()
        k = int(((base["vix_z"] > z) & (base["spx_z"].abs() < f)).sum())
        cnt.loc[f"z>{z}", f"|s|<{f}"] = k
        rat.loc[f"z>{z}", f"|s|<{f}"] = round(k / (ps * pf * n), 3)
print(cnt.astype(int).to_string())
print("\n[observed / expected under independence]")
print(rat.to_string())

sig_idx = base.index[SIG]
mag = pd.DataFrame({
    "vix_prev": px["VIX"].shift(1), "vix_pt": px["VIX"].diff(),
    "vix_pct": px["VIX"].pct_change() * 100, "vix_z": d["vix_z"],
    "decade": px.index.year // 10 * 10}).loc[sig_idx]
print("\n[what the signal actually is: median by decade]")
print(mag.groupby("decade")[["vix_prev", "vix_pt", "vix_pct", "vix_z"]].median().round(2).to_string())
print("VIX move < 5%:", int((mag["vix_pct"] < 5).sum()), "/", len(mag))

dow = pd.DataFrame({
    "all": pd.Series(base.index.dayofweek).value_counts(normalize=True).sort_index(),
    "hi_flat": pd.Series(sig_idx.dayofweek).value_counts(normalize=True).sort_index(),
    "hi_move": pd.Series(base.index[grp == "hi_move"].dayofweek).value_counts(normalize=True).sort_index(),
}).round(3)
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri"]
print("\n[day-of-week share]")
print(dow.to_string())

print("\n[signals per decade]")
print(pd.Series(sig_idx.year // 10 * 10).value_counts().sort_index().to_string())

[sample] 1990-03-01 ~ 2026-07-24 | n = 9166

[2x2 groups]
lo_flat    6373
lo_move    2154
hi_move     502
hi_flat     137

[signal counts by threshold]
        |s|<0.25  |s|<0.5  |s|<0.75  |s|<1.0
z>1.5         21       45        79      137
z>1.75        11       24        39       72
z>2.0          2        9        16       32
z>2.5          0        2         5       11

[observed / expected under independence]
        |s|<0.25  |s|<0.5  |s|<0.75  |s|<1.0
z>1.5      0.137    0.158     0.208    0.302
z>1.75     0.101    0.119     0.145    0.224
z>2.0      0.025    0.059     0.079    0.133
z>2.5      0.000    0.023     0.044    0.080

[what the signal actually is: median by decade]
        vix_prev  vix_pt  vix_pct  vix_z
decade                                  
1990       15.51    1.53     9.02   1.76
2000       17.61    1.36     8.35   1.74
2010       14.03    1.84    11.68   1.85
2020       18.51    2.05    10.16   1.82
VIX move < 5%: 1 / 137

[day-of-week share]
       all  hi_fl

In [19]:
# 4

fwd = pd.DataFrame(index=base.index)
for h in HORIZONS:
    fwd[f"h{h}"] = d["spx_ret"].shift(-h).reindex(base.index)
fwd["cum1_5"] = fwd[[f"h{h}" for h in HORIZONS]].sum(axis=1, min_count=len(HORIZONS))

print("[forward SPX returns, bps: hi_flat vs lo_flat]")
rows = []
for c in fwd.columns:
    r = compare(fwd[c], SIG, BASE, scale=1e4)
    rows.append({"horizon": c, **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[all four groups, mean bps]")
tab = pd.DataFrame({c: (fwd[c].groupby(grp).mean() * 1e4).round(1) for c in fwd.columns})
tab["n"] = grp.value_counts()
print(tab.loc[["hi_flat", "hi_move", "lo_flat", "lo_move"]].to_string())

print("\n[distribution of h1, h2: selection vs convexity]")
rows = []
for c in ["h1", "h2"]:
    for g in ["hi_flat", "lo_flat"]:
        x = fwd.loc[(grp == g).values, c].dropna()
        rows.append({"horizon": c, "group": g, "n": len(x),
                     "pct_neg": round((x < 0).mean() * 100, 1),
                     "q05": round(x.quantile(0.05) * 1e4, 1),
                     "q25": round(x.quantile(0.25) * 1e4, 1),
                     "median": round(x.median() * 1e4, 1),
                     "q75": round(x.quantile(0.75) * 1e4, 1),
                     "q95": round(x.quantile(0.95) * 1e4, 1)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[regime split, bps]")
rows = []
yrs = base.index.year
for name, y0, y1 in PERIODS:
    w = (yrs >= y0) & (yrs <= y1)
    for c in ["h1", "h2", "cum1_5"]:
        s = fwd.loc[w & SIG, c].dropna()
        b = fwd.loc[w & BASE, c].dropna()
        if len(s) < 3:
            continue
        ma, mb, t, na, nb = welch(s, b)
        rows.append({"period": name, "horizon": c, "n_sig": na,
                     "signal": round(ma * 1e4, 1), "lo_flat": round(mb * 1e4, 1),
                     "diff": round((ma - mb) * 1e4, 1), "t_welch": round(t, 2)})
print(pd.DataFrame(rows).to_string(index=False))

[forward SPX returns, bps: hi_flat vs lo_flat]
horizon  n_sig  n_base  signal  base  diff  t_welch  perm_z  p_perm
     h1    137    6372   11.66  2.40  9.25     1.23    0.88    0.38
     h2    137    6372   17.09  2.46 14.63     1.75    1.42    0.15
     h3    137    6371    1.36  3.34 -1.98    -0.22   -0.21    0.84
     h4    137    6371    4.36  3.09  1.27     0.15    0.10    0.93
     h5    137    6370   -0.62  3.65 -4.27    -0.59   -0.42    0.68
 cum1_5    137    6370   33.84 14.97 18.87     1.07    0.84    0.39

[all four groups, mean bps]
           h1    h2   h3    h4   h5  cum1_5     n
hi_flat  11.7  17.1  1.4   4.4 -0.6    33.8   137
hi_move  15.3  -1.4  6.9  11.4  8.4    40.6   502
lo_flat   2.4   2.5  3.3   3.1  3.7    15.0  6373
lo_move   3.0   6.3  2.8   2.3  1.6    16.1  2154

[distribution of h1, h2: selection vs convexity]
horizon   group    n  pct_neg    q05   q25  median  q75   q95
     h1 hi_flat  137     46.0 -134.2 -39.7    14.7 68.7 145.8
     h1 lo_flat 6372    

In [20]:
# 5

sec_ret = np.log(px[SECTORS]).diff()
r9 = np.log(px[SEC9]).diff()

rot_raw = pd.DataFrame(index=px.index)
rot_raw["disp"] = np.log(sec_ret.std(axis=1, ddof=1))
rot_raw["rankcorr"] = sec_ret.corrwith(sec_ret.shift(1), axis=1, method="spearman")
rot_raw["n_sec"] = sec_ret.notna().sum(axis=1)

ctrl = pd.DataFrame({"y": rot_raw["disp"], "absz": d["spx_z"].abs(),
                     "n9": (rot_raw["n_sec"] == 9).astype(float),
                     "n10": (rot_raw["n_sec"] == 10).astype(float)}).dropna()
Xc = np.column_stack([np.ones(len(ctrl)), ctrl["absz"], ctrl["n9"], ctrl["n10"]])
yc = ctrl["y"].values
resid = np.full(len(ctrl), np.nan)
for i in range(ROLL_W, len(ctrl)):
    c, *_ = np.linalg.lstsq(Xc[i - ROLL_W:i], yc[i - ROLL_W:i], rcond=None)
    resid[i] = yc[i] - Xc[i] @ c
rot_raw["disp_resid"] = pd.Series(resid, index=ctrl.index)

print("[control regression: log(disp) ~ |spx_z| + n_sec dummies, rolling 750d]")
cf, *_ = np.linalg.lstsq(Xc, yc, rcond=None)
print("full-sample coef [const, |z|, n9, n10]:", np.round(cf, 4),
      "| R2:", round(1 - np.var(yc - Xc @ cf) / np.var(yc), 4))

print("\n[h1: hi_flat vs lo_flat]")
rows = []
for m in ["disp", "disp_resid", "rankcorr"]:
    y1 = rot_raw[m].shift(-1).reindex(base.index)
    r = compare(y1, SIG, BASE)
    rows.append({"measure": m, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[disp_resid across horizons]")
rows = []
for h in [0] + HORIZONS:
    y = rot_raw["disp_resid"].shift(-h).reindex(base.index)
    r = compare(y, SIG, BASE)
    rows.append({"h": h, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[persistence control: is h1 just carryover from h0?]")
rr0 = rot_raw["disp_resid"].reindex(base.index)
rr1 = rot_raw["disp_resid"].shift(-1).reindex(base.index)
ar = pd.DataFrame({"y": rr1, "x": rr0}).dropna()
sl, ic = np.polyfit(ar["x"].values, ar["y"].values, 1)
print(f"AR(1) slope: {sl:.4f} | corr(h0,h1): {ar['x'].corr(ar['y']):.3f}")
rows = []
for lbl, y in [("h1 raw", rr1), ("h1 | h0", rr1 - (sl * rr0 + ic)), ("h1 - h0", rr1 - rr0)]:
    r = compare(y, SIG, BASE)
    rows.append({"spec": lbl, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

[control regression: log(disp) ~ |spx_z| + n_sec dummies, rolling 750d]
full-sample coef [const, |z|, n9, n10]: [-5.0381  0.1708 -0.0778 -0.2786] | R2: 0.0779

[h1: hi_flat vs lo_flat]
   measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
      disp     92    4843 -5.1464 -4.9955 -0.1509  -2.7217 -3.1505  0.0018
disp_resid     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.0461  0.0000
  rankcorr     92    4842 -0.0397 -0.0275 -0.0122  -0.2792 -0.3062  0.7766

[disp_resid across horizons]
 h  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
 0     79    4323 -0.1227 -0.0555 -0.0672  -1.2580 -1.4696  0.1396
 1     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.1106  0.0000
 2     79    4324 -0.1289 -0.0649 -0.0640  -1.3209 -1.5427  0.1246
 3     79    4324 -0.1467 -0.0609 -0.0858  -1.7270 -1.9422  0.0520
 4     79    4324 -0.0987 -0.0662 -0.0325  -0.6838 -0.9334  0.3556
 5     79    4324 -0.1619 -0.0624 -0.0995  -2.0137 -2.3037  0.0202

[persistence co

In [23]:
# 6

y_rank = rot_raw["rankcorr"].shift(-1).reindex(base.index)
y_disp = rot_raw["disp_resid"].shift(-1).reindex(base.index)

VOLS = [("vix", "VIX"), ("ovx", "OVX"), ("gvz", "GVZ")]


def vol_masks(v, window=None):
    zz = d[f"{v}_z"].reindex(base.index)
    ok = zz.notna().values
    if window is not None:
        ok = ok & window
    return ((zz > Z) & fl).values & ok, ((zz <= Z) & fl).values & ok


print("[primary: each vol index on its own full history]")
rows = []
for v, V in VOLS:
    s, b = vol_masks(v)
    for lbl, y in [("rankcorr", y_rank), ("disp_resid", y_disp)]:
        ma, mb, t, na, nb = welch(y[s], y[b])
        _, sd, pz, pp = perm_diff(y.values, s)
        rows.append({"vol": V, "since": base.index[s].min().year, "measure": lbl,
                     "n_sig": na, "n_base": nb, "signal": round(ma, 4), "base": round(mb, 4),
                     "diff": round(ma - mb, 4), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[forward SPX returns, bps]")
rows = []
for v, V in VOLS:
    s, b = vol_masks(v)
    for c in ["h1", "h2"]:
        ma, mb, t, na, nb = welch(fwd[c][s], fwd[c][b])
        _, _, pz, pp = perm_diff(fwd[c].values, s)
        rows.append({"vol": V, "horizon": c, "n_sig": na,
                     "signal": round(ma * 1e4, 1), "base": round(mb * 1e4, 1),
                     "diff": round((ma - mb) * 1e4, 1), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[diagnostic: why the 2008+ common sample was discarded]")
print("disp_resid, VIX only")
rows = []
for lbl, win in [("full (1990+)", None), ("common (2008+)", w08)]:
    s, b = vol_masks("vix", win)
    yy = y_disp.copy()
    if win is not None:
        yy[~win] = np.nan
    ma, mb, t, na, nb = welch(yy[s], yy[b])
    _, sd, pz, pp = perm_diff(yy.values, s)
    rows.append({"sample": lbl, "n_sig": na, "diff": round(ma - mb, 4),
                 "se_welch": round((ma - mb) / t, 4), "perm_sd": round(sd, 4),
                 "t_welch": round(t, 2), "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
dg = pd.DataFrame(rows)
print(dg.to_string(index=False))
print(f"\nperm_sd ratio (common / full): {dg['perm_sd'].iloc[1] / dg['perm_sd'].iloc[0]:.1f}x"
      f"  vs  se_welch ratio: {dg['se_welch'].iloc[1] / dg['se_welch'].iloc[0]:.1f}x")
print("effect size is unchanged; the circular-shift null distribution destabilises on the shorter window.")

[primary: each vol index on its own full history]
vol  since    measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
VIX   1990   rankcorr     92    4842 -0.0397 -0.0275 -0.0122    -0.28   -0.31  0.7632
VIX   1990 disp_resid     79    4323 -0.2474 -0.0672 -0.1803    -3.29   -4.13  0.0000
OVX   2007   rankcorr    179    3251 -0.0420 -0.0238 -0.0182    -0.57   -0.39  0.7010
OVX   2007 disp_resid    179    3251  0.0120 -0.0271  0.0391     1.05    0.90  0.3710
GVZ   2008   rankcorr    192    3060 -0.0340 -0.0190 -0.0149    -0.49   -0.17  0.8614
GVZ   2008 disp_resid    192    3060 -0.0312 -0.0487  0.0175     0.48    0.36  0.7298

[forward SPX returns, bps]
vol horizon  n_sig  signal  base  diff  t_welch  perm_z  p_perm
VIX      h1    137    11.7   2.4   9.3     1.23    0.89  0.3724
VIX      h2    137    17.1   2.5  14.6     1.75    1.40  0.1614
OVX      h1    179     2.0   3.5  -1.5    -0.13   -0.15  0.8822
OVX      h2    179    17.5   2.8  14.6     1.43    1.69  0.0928


In [24]:
#7a
lr_spx = np.log(px["SPX"]).diff()
lr_vix = np.log(px["VIX"]).diff()

print("[index and alignment]")
print("monotonic:", px.index.is_monotonic_increasing,
      "| duplicates:", int(px.index.duplicated().sum()),
      "| d aligned to px:", d.index.equals(px.index),
      "| base subset of d:", base.index.isin(d.index).all())

errs_s, errs_v = [], []
for ti in rng.choice(np.arange(WIN + 5, len(px)), 100, replace=False):
    hs, hv = lr_spx.iloc[ti - WIN:ti], lr_vix.iloc[ti - WIN:ti]
    if hs.notna().sum() >= MINP:
        errs_s.append(abs(lr_spx.iloc[ti] / hs.std(ddof=1) - d["spx_z"].iloc[ti]))
    if hv.notna().sum() >= MINP:
        errs_v.append(abs(lr_vix.iloc[ti] / hv.std(ddof=1) - d["vix_z"].iloc[ti]))
print("\n[look-ahead: manual recompute of z-scores, 100 random dates]")
print("max abs err spx_z:", np.nanmax(errs_s), "| vix_z:", np.nanmax(errs_v))

pos = d.index.get_indexer(base.index)
bad = 0
for h in HORIZONS:
    tgt = np.where(pos + h < len(d), d["spx_ret"].values[np.clip(pos + h, 0, len(d) - 1)], np.nan)
    bad += int((np.abs(np.nan_to_num(tgt) - np.nan_to_num(fwd[f"h{h}"].values)) > 1e-12).sum())
print("\n[forward alignment]")
print("misaligned cells:", bad)
print("cum1_5 == sum(h1..h5):",
      bool(np.nanmax(np.abs(fwd[[f"h{h}" for h in HORIZONS]].sum(axis=1, min_count=5) - fwd["cum1_5"])) < 1e-12))

ridx = rot_raw.index
chk = []
for k in rng.choice(np.arange(ROLL_W + 10, len(ctrl)), 5, replace=False):
    dt = ctrl.index[k]
    c, *_ = np.linalg.lstsq(Xc[k - ROLL_W:k], yc[k - ROLL_W:k], rcond=None)
    chk.append(abs((yc[k] - Xc[k] @ c) - rot_raw.loc[dt, "disp_resid"]))
print("\n[rolling residual uses only past window]")
print("max abs err:", max(chk))
print("first valid disp_resid:", rot_raw['disp_resid'].first_valid_index().date(),
      "| ctrl start:", ctrl.index.min().date())

print("\n[degenerate observations]")
print("zero dlog VIX days:", int((lr_vix == 0).sum()),
      "| among signals:", int((lr_vix.reindex(base.index[SIG]) == 0).sum()))
print("zero dlog SPX days:", int((lr_spx == 0).sum()))
print("signal days with any sector NaN:",
      int(sec_ret.reindex(base.index[SIG]).isna().any(axis=1).sum()), "/", int(SIG.sum()))

print("\n[signal clustering]")
s_ser = pd.Series(SIG.astype(int), index=base.index)
gaps = np.diff(np.where(SIG)[0])
print("autocorr lag1-5:", [round(s_ser.autocorr(l), 3) for l in range(1, 6)])
print("gap median:", int(np.median(gaps)), "| gaps <= 5d:", int((gaps <= 5).sum()), "/", len(gaps),
      "| max consecutive:", int(s_ser.groupby((s_ser != s_ser.shift()).cumsum()).cumsum().max()))

[index and alignment]
monotonic: True | duplicates: 0 | d aligned to px: True | base subset of d: True

[look-ahead: manual recompute of z-scores, 100 random dates]
max abs err spx_z: 3.68594044175552e-14 | vix_z: 4.440892098500626e-15

[forward alignment]
misaligned cells: 0
cum1_5 == sum(h1..h5): True

[rolling residual uses only past window]
max abs err: 0.0
first valid disp_resid: 2001-12-19 | ctrl start: 1998-12-23

[degenerate observations]
zero dlog VIX days: 51 | among signals: 0
zero dlog SPX days: 5
signal days with any sector NaN: 109 / 137

[signal clustering]
autocorr lag1-5: [np.float64(-0.008), np.float64(-0.0), np.float64(0.022), np.float64(0.014), np.float64(0.029)]
gap median: 54 | gaps <= 5d: 18 / 136 | max consecutive: 2


In [25]:
#7b
print("[A. threshold surface: hi_flat vs lo_flat at each (Z, F)]")
rows = []
for zt in [1.25, 1.5, 1.75, 2.0]:
    for ft in [0.5, 0.75, 1.0, 1.25]:
        s = ((base["vix_z"] > zt) & (base["spx_z"].abs() < ft)).values
        b = ((base["vix_z"] <= zt) & (base["spx_z"].abs() < ft)).values
        if s.sum() < 10:
            continue
        mr, br, tr, nr, _ = welch(fwd["h1"][s], fwd["h1"][b])
        md, bd, td, nd, _ = welch(y_disp[s], y_disp[b])
        _, _, pzd, ppd = perm_diff(y_disp.values, s)
        rows.append({"Z": zt, "F": ft, "n_ret": nr, "h1_diff_bps": round((mr - br) * 1e4, 1),
                     "t_ret": round(tr, 2), "n_disp": nd, "disp_diff": round(md - bd, 4),
                     "t_disp": round(td, 2), "perm_z": round(pzd, 2), "p_perm": round(ppd, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. day-of-week: signal is 45% Monday]")
dm = pd.Series({k: rot_raw["disp_resid"].reindex(base.index)[base.index.dayofweek == i].mean()
                for i, k in enumerate(["Mon", "Tue", "Wed", "Thu", "Fri"])})
print("mean disp_resid by weekday:", dm.round(4).to_dict())

nm = base.index.dayofweek != 0
rows = []
for lbl, m in [("all days", np.ones(len(base), dtype=bool)), ("Mon excluded", nm)]:
    yy = y_disp.copy()
    yy[~m] = np.nan
    r = compare(yy, SIG & m, BASE & m)
    rows.append({"spec": lbl, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})

lv_dm = lr_vix - lr_vix.groupby(lr_vix.index.dayofweek).transform("mean")
z_dm = (lv_dm / lv_dm.rolling(WIN, min_periods=MINP).std().shift(1)).reindex(base.index)
sig_dm = ((z_dm > Z) & fl).values
r = compare(y_disp, sig_dm, BASE)
rows.append({"spec": "DOW-demeaned VIX", **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print()
print(pd.DataFrame(rows).to_string(index=False))
print("Monday share after demeaning:", round((base.index[sig_dm].dayofweek == 0).mean(), 3),
      "| original:", round((base.index[SIG].dayofweek == 0).mean(), 3))

print("\n[C. z-matched: flat vs move within narrow vix_z bands]")
rows = []
for lo, hiz in [(1.5, 1.8), (1.8, 2.1), (2.1, 2.6), (2.6, 99)]:
    inb = (base["vix_z"] > lo) & (base["vix_z"] <= hiz)
    a, b = (inb & fl).values, (inb & ~fl).values
    med_a = base.loc[a, "vix_z"].median()
    med_b = base.loc[b, "vix_z"].median()
    md, bd, td, nd, nb = welch(y_disp[a], y_disp[b])
    if nd < 5 or nb < 5:
        continue
    rows.append({"band": f"{lo}-{hiz}", "n_flat": nd, "n_move": nb,
                 "medz_flat": round(med_a, 2), "medz_move": round(med_b, 2),
                 "flat": round(md, 4), "move": round(bd, 4),
                 "diff": round(md - bd, 4), "t_welch": round(td, 2)})
print(pd.DataFrame(rows).to_string(index=False))

[A. threshold surface: hi_flat vs lo_flat at each (Z, F)]
   Z    F  n_ret  h1_diff_bps  t_ret  n_disp  disp_diff  t_disp  perm_z  p_perm
1.25 0.50     86         -0.7  -0.08      31    -0.3556   -4.05   -4.83  0.0000
1.25 0.75    147         -1.6  -0.25      72    -0.2041   -3.32   -4.31  0.0000
1.25 1.00    234          1.8   0.31     135    -0.1209   -2.81   -3.49  0.0000
1.25 1.25    332         -1.2  -0.23     196    -0.0901   -2.63   -3.15  0.0016
1.50 0.50     45          1.1   0.10      15    -0.3597   -3.06   -4.07  0.0002
1.50 0.75     79         10.0   1.21      37    -0.3186   -3.91   -5.00  0.0000
1.50 1.00    137          9.3   1.23      79    -0.1803   -3.29   -4.09  0.0000
1.50 1.25    201          3.1   0.44     118    -0.1276   -3.00   -3.55  0.0008
1.75 0.50     24         13.2   0.85      11    -0.3618   -2.44   -3.17  0.0004
1.75 0.75     39         14.5   1.22      21    -0.3679   -3.75   -4.06  0.0000
1.75 1.00     72         11.1   1.27      44    -0.2436   -3.8

In [26]:
#7c
alt = pd.DataFrame(index=px.index)
alt["disp_resid"] = rot_raw["disp_resid"]
alt["sd9"] = np.log(r9.std(axis=1, ddof=1))
alt["mad9"] = np.log(r9.sub(r9.mean(axis=1), axis=0).abs().mean(axis=1))
bt = sec_ret.rolling(250, min_periods=150).cov(d["spx_ret"]).div(
    d["spx_ret"].rolling(250, min_periods=150).var(), axis=0).shift(1)
alt["beta_disp"] = np.log((sec_ret - bt.mul(d["spx_ret"], axis=0)).std(axis=1, ddof=1))
alt["rankcorr"] = rot_raw["rankcorr"]

print("[A. alternative rotation measures, h1]")
rows = []
for c in alt.columns:
    y1 = alt[c].shift(-1).reindex(base.index)
    r = compare(y1, SIG, BASE)
    rows.append({"measure": c, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. outlier robustness: disp_resid h1]")
a, b = y_disp[SIG].dropna(), y_disp[BASE].dropna()
lo_, hi_ = y_disp.quantile([0.01, 0.99])
print(f"mean diff      : {a.mean() - b.mean():.4f}")
print(f"median diff    : {a.median() - b.median():.4f}")
print(f"winsorised 1/99: {a.clip(lo_, hi_).mean() - b.clip(lo_, hi_).mean():.4f}")
infl = (a - a.mean()).abs().sort_values(ascending=False)
print("drop most influential signals:",
      {f"top{k}": round(a.drop(infl.index[:k]).mean() - b.mean(), 4) for k in [1, 3, 5, 10]})

print("\n[C. multiplicity across horizons h0-h5, disp_resid]")
H = [0] + HORIZONS
Y = np.column_stack([alt["disp_resid"].shift(-h).reindex(base.index).values for h in H])
obs = np.array([np.nanmean(Y[SIG & ~np.isnan(Y[:, j]), j]) - np.nanmean(Y[~SIG & ~np.isnan(Y[:, j]), j])
                for j in range(Y.shape[1])])
nrow = len(base)
null = np.empty((N_PERM, len(H)))
for i in range(N_PERM):
    sh = np.roll(SIG, rng.integers(1, nrow))
    for j in range(len(H)):
        y = Y[:, j]
        ok = ~np.isnan(y)
        null[i, j] = np.nanmean(y[sh & ok]) - np.nanmean(y[~sh & ok])
mu, sd = null.mean(0), null.std(0)
zobs = (obs - mu) / sd
znull = np.abs((null - mu) / sd)
print("per-horizon:", pd.DataFrame({"h": H, "diff": obs.round(4), "z": zobs.round(2),
                                    "p_indiv": [(znull[:, j] >= abs(zobs[j])).mean() for j in range(len(H))]
                                    }).to_string(index=False))
print(f"\nfamily-wise p (max |z| across 6 horizons): {(znull.max(1) >= np.abs(zobs).max()).mean():.4f}")

print("\n[D. sub-period stability: disp_resid h1]")
rows = []
for name, y0, y1_ in PERIODS:
    w = (base.index.year >= y0) & (base.index.year <= y1_)
    s_, b_ = y_disp[w & SIG].dropna(), y_disp[w & BASE].dropna()
    if len(s_) < 5:
        continue
    ma, mb, t, na, nb = welch(s_, b_)
    rows.append({"period": name, "n_sig": na, "signal": round(ma, 4),
                 "lo_flat": round(mb, 4), "diff": round(ma - mb, 4), "t_welch": round(t, 2)})
print(pd.DataFrame(rows).to_string(index=False))

[A. alternative rotation measures, h1]
   measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
disp_resid     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.0500  0.0000
       sd9     92    4843 -5.1567 -4.9996 -0.1570  -2.7597 -3.2483  0.0012
      mad9     92    4843 -5.4257 -5.2842 -0.1415  -2.4909 -2.9957  0.0024
 beta_disp     88    4745 -5.2285 -5.0673 -0.1611  -2.8909 -3.4532  0.0004
  rankcorr     92    4842 -0.0397 -0.0275 -0.0122  -0.2792 -0.3023  0.7654

[B. outlier robustness: disp_resid h1]
mean diff      : -0.1803
median diff    : -0.1278
winsorised 1/99: -0.1790
drop most influential signals: {'top1': np.float64(-0.1981), 'top3': np.float64(-0.2262), 'top5': np.float64(-0.2015), 'top10': np.float64(-0.1696)}

[C. multiplicity across horizons h0-h5, disp_resid]
per-horizon:  h    diff     z  p_indiv
 0 -0.0700 -1.47   0.1452
 1 -0.1964 -4.06   0.0000
 2 -0.0763 -1.64   0.0986
 3 -0.0944 -1.94   0.0526
 4 -0.0458 -0.93   0.3612
 5 -0.1097 -2.30   0.0216

In [27]:
#3b
END = base.index.max()
W2M = base.index >= (END - pd.DateOffset(months=2))
print("[recent window]", base.index[W2M].min().date(), "~", END.date(),
      "| trading days:", int(W2M.sum()))

zs = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
fs = [0.75, 1.0, 1.5, 2.0]

print("\n[signal count in recent 2 months]")
cnt2 = pd.DataFrame(index=[f"z>{z}" for z in zs], columns=[f"|s|<{f}" for f in fs], dtype=int)
for z in zs:
    for f in fs:
        cnt2.loc[f"z>{z}", f"|s|<{f}"] = int(((base["vix_z"] > z) & (base["spx_z"].abs() < f) & W2M).sum())
print(cnt2.to_string())

print("\n[implied full-sample frequency, signals per year]")
freq = pd.DataFrame(index=[f"z>{z}" for z in zs], columns=[f"|s|<{f}" for f in fs], dtype=float)
yrs_total = len(base) / 252
for z in zs:
    for f in fs:
        k = ((base["vix_z"] > z) & (base["spx_z"].abs() < f)).sum()
        freq.loc[f"z>{z}", f"|s|<{f}"] = round(k / yrs_total, 1)
print(freq.to_string())

print("\n[what each z threshold means: percentile of daily VIX moves, and median move]")
rows = []
for z in zs:
    m = base["vix_z"] > z
    pct = 100 * (1 - m.mean())
    mv = d["vix_ch"].reindex(base.index)[m]
    rows.append({"z": z, "pctile": round(pct, 1), "n_days": int(m.sum()),
                 "median_pct_move": round((np.exp(mv.median()) - 1) * 100, 2),
                 "min_pct_move": round((np.exp(mv.min()) - 1) * 100, 2)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[recent 2 months, day by day: all days with any VIX increase]")
rec = pd.DataFrame({
    "VIX": px["VIX"].reindex(base.index),
    "vix_pct": (np.exp(d["vix_ch"]) - 1).reindex(base.index) * 100,
    "vix_z": base["vix_z"],
    "spx_pct": (np.exp(d["spx_ret"]) - 1).reindex(base.index) * 100,
    "spx_z": base["spx_z"],
})[W2M]
rec = rec[rec["vix_pct"] > 0].sort_values("vix_z", ascending=False)
print(rec.round(2).to_string())

[recent window] 2026-05-26 ~ 2026-07-24 | trading days: 42

[signal count in recent 2 months]
        |s|<0.75  |s|<1.0  |s|<1.5  |s|<2.0
z>0.25       6.0      8.0     11.0     13.0
z>0.5        5.0      6.0      9.0     11.0
z>0.75       1.0      2.0      5.0      7.0
z>1.0        0.0      1.0      4.0      6.0
z>1.25       0.0      1.0      4.0      6.0
z>1.5        0.0      1.0      4.0      5.0
z>2.0        0.0      0.0      0.0      0.0

[implied full-sample frequency, signals per year]
        |s|<0.75  |s|<1.0  |s|<1.5  |s|<2.0
z>0.25      43.9     54.8     70.7     79.3
z>0.5       27.0     35.1     49.0     57.1
z>0.75      14.7     20.5     31.4     38.7
z>1.0        8.2     11.9     19.8     26.0
z>1.25       4.0      6.4     11.7     16.9
z>1.5        2.2      3.8      7.1     10.9
z>2.0        0.4      0.9      2.1      4.2

[what each z threshold means: percentile of daily VIX moves, and median move]
   z  pctile  n_days  median_pct_move  min_pct_move
0.25    64.9    3213